In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Setup paths relative to the notebooks folder
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "eep_ecc_integrated_metrics.csv"

# Fallback path if working directory is the project root
if not PROCESSED_DATA_PATH.exists():
    PROCESSED_DATA_PATH = Path("data/processed/eep_ecc_integrated_metrics.csv")

# Load integrated data
df = pd.read_csv(PROCESSED_DATA_PATH)
print("📦 Integrated multi-agency data loaded successfully!")
display(df)

📦 Integrated multi-agency data loaded successfully!


,Fiscal_Year,Era,Total_Revenue_Billion_Birr,Net_Income_Loss_Billion_Birr,Total_FX_Revenue_Million_USD,Data_Mining_FX_Contribution_Million_USD,Regional_Export_FX_Contribution_Million_USD,Total_Grid_Generation_GWh,Data_Mining_Consumption_GWh,Data_Mining_Grid_Share_Pct,Est_Hardware_Import_Value_Million_USD,Avg_Customs_Duty_Tariff_Pct,Upfront_Customs_Revenue_Million_USD,Leading_Hardware_Import_Value_Next_Year_FDI
0,2021/22,Pre-Data Mining,18.5,-26.2,112.0,0.0,112.0,15400,0,0.0,0.0,5.0,0.000,8.5
1,2022/23,Pre-Data Mining,22.0,-24.8,125.0,0.0,125.0,18200,0,0.0,8.5,5.0,0.425,55.0
2,2023/24,Pre-Data Mining Transition,27.0,-10.0,140.0,15.0,125.0,22100,1100,5.0,55.0,5.0,2.750,195.0
3,2024/25,Post-Data Mining Era,75.4,5.2,338.0,220.0,118.0,29480,7960,27.0,195.0,5.0,9.750,260.0
4,2025/26 (Proj),Post-Data Mining Era,109.0,14.2,410.0,285.0,125.0,34500,10350,30.0,260.0,5.0,13.000,NaN


In [2]:
# 1. Prepare features and target
# We exclude the final row for training because we don't have the "next year" target for it yet
train_df = df.dropna(subset=['Leading_Hardware_Import_Value_Next_Year_FDI']).copy()

# Feature: Current Hardware Import Value (Million USD)
X_train = train_df[['Est_Hardware_Import_Value_Million_USD']]
# Target: Next year's Data Mining FX Revenue contribution to EEP
y_train = train_df['Data_Mining_FX_Contribution_Million_USD']

# 2. Train a Linear Regression Model to discover the scaling coefficient
model = LinearRegression()
model.fit(X_train, y_train)

# Calculate metrics on our historical data points
y_pred = model.predict(X_train)
mae = mean_absolute_error(y_train, y_pred)
r2 = r2_score(y_train, y_pred)

print("🎯 ML Model Training Complete!")
print(f"📈 Model R² Score (Fit Quality): {r2:.4f}")
print(f"💵 Mean Absolute Error: ${mae:.2f} Million USD")
print(f"⚡ Discovered Impact Metric: For every $1 USD of hardware clearing Customs, "
      f"EEP's future annual FX revenue scales by approximately ${model.coef_[0]:.2f} USD!")

🎯 ML Model Training Complete!
📈 Model R² Score (Fit Quality): 0.9580
💵 Mean Absolute Error: $16.25 Million USD
⚡ Discovered Impact Metric: For every $1 USD of hardware clearing Customs, EEP's future annual FX revenue scales by approximately $1.17 USD!


In [4]:
# 3. Predict the future using our leading indicator feature
# We take the latest hardware import value (e.g., 2025/26 Proj) to forecast the upcoming fiscal year's FX contribution
latest_import_value = df.iloc[-1]['Est_Hardware_Import_Value_Million_USD']
future_feature = np.array([[latest_import_value]])

predicted_future_fx = model.predict(future_feature)[0]

print(f"\n🔮 --- FUTURE GRID FORECAST ---")
print(f"Based on current hardware import trends passing through customs (${latest_import_value}M USD),")
print(f"The model predicts EEP's Data Mining FX Revenue will scale to: **${predicted_future_fx:.1f} Million USD** in the next cycle!")



🔮 --- FUTURE GRID FORECAST ---
Based on current hardware import trends passing through customs ($260.0M USD),
The model predicts EEP's Data Mining FX Revenue will scale to: **$287.1 Million USD** in the next cycle!


c:\Users\umers\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


### 🏛️ Multi-Agency Impact Analysis: Fiscal vs. Utility Revenues

It is crucial to distinguish between the two distinct revenue streams generated by the expansion of data centers:

1. **Utility Revenue (EEP):** Collected directly by Ethiopian Electric Power via energy billing. This hard currency ($220M+ USD) is retained by the utility to maintain turbines, build substations, and expand the national grid.
2. **Fiscal Sovereign Revenue (Customs/Government):** Collected upfront at entry hubs like the Kality Customs Hub by the Ethiopian Customs Commission (ECC). These duties and processing fees go directly into the federal treasury, immediately benefiting the federal budget to fund public infrastructure, health, and education.

By modeling both, this data pipeline proves that tech infrastructure investments provide a **dual benefit**: an immediate cash injection to the sovereign government at the border, followed by a long-term, continuous revenue stream for the state utility once the hardware goes live on the grid.

In [5]:
import plotly.graph_objects as ob

# Calculate cumulative totals to show the macro impact over the era
total_customs_gov = df['Upfront_Customs_Revenue_Million_USD'].sum()
total_eep_util = df['Data_Mining_FX_Contribution_Million_USD'].sum()

fig_gov = ob.Figure(data=[
    ob.Bar(
        x=['Federal Treasury (Customs Revenue)', 'EEP Balance Sheet (Utility FX)'],
        y=[total_customs_gov, total_eep_util],
        marker_color=['#1abc9c', '#3498db'],
        text=[f"${total_customs_gov:.1f}M USD", f"${total_eep_util:.1f}M USD"],
        textposition='auto'
    )
])

fig_gov.update_layout(
    title_text="<b>Dual-Agency Inflow Breakdown (Cumulative Portfolio Impact)</b>",
    yaxis_title="Total Inflow (Million USD)",
    template="plotly_white",
    height=450,
    width=700
)

fig_gov.show()